# GARCH 模型分析 - 虛擬貨幣波動率預測

本筆記本展示如何使用 GARCH (Generalized Autoregressive Conditional Heteroskedasticity) 模型來預測虛擬貨幣的波動率。

## 學習目標
1. 理解 GARCH 模型的基本概念與原理
2. 學會檢測 ARCH 效應
3. 掌握波動率預測技術
4. 計算風險指標 (VaR, CVaR)
5. 分析波動率聚集現象

## GARCH 模型簡介

GARCH 模型是專門用來分析和預測金融時間序列波動率的模型：

**基本 GARCH(1,1) 模型**：
- 收益率方程: $r_t = \mu + \varepsilon_t$
- 波動率方程: $\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$

其中：
- $\omega$ = 長期波動率水準
- $\alpha$ = ARCH 項係數（短期衝擊影響）
- $\beta$ = GARCH 項係數（波動率持續性）

In [ ]:
# 導入必要的套件
import sys
import os
sys.path.append('../../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 中文字體設定
try:
    plt.rcParams['font.family'] = 'Microsoft YaHei'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 設定圖表樣式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# 導入 GARCH 模型和數據收集器
from models.foundations.garch_model import GARCHModel
from data_collection.data_downloader import UnifiedDataDownloader

print("✅ 套件導入完成！")

## 1. 數據準備與收集

我們將使用真實的 BTC 交易數據來進行 GARCH 分析。

In [ ]:
# 初始化數據下載器
downloader = UnifiedDataDownloader(verify_ssl=False)

# 下載 BTC 數據 (使用4小時K線，過去60天)
print("📥 正在下載 BTC 數據...")
btc_data = downloader.create_dataset_for_arima(
    symbol='BTCUSDT',
    interval='4h',
    days=60,
    exchange='binance'
)

if btc_data.empty:
    print("⚠️ 無法獲取真實數據，使用模擬數據")
    # 生成模擬數據
    dates = pd.date_range('2024-01-01', periods=360, freq='4H')
    np.random.seed(42)
    
    # 模擬 GARCH 過程
    omega, alpha, beta = 0.1, 0.1, 0.8
    n = len(dates)
    returns = np.zeros(n)
    volatility = np.zeros(n)
    volatility[0] = 2.0  # 初始波動率 2%
    
    for t in range(1, n):
        volatility[t] = np.sqrt(omega + alpha * returns[t-1]**2 + beta * volatility[t-1]**2)
        returns[t] = volatility[t] * np.random.normal(0, 1)
    
    # 轉換為價格數據
    prices = 50000 * np.exp(np.cumsum(returns / 100))
    
    btc_data = pd.DataFrame({
        'close': prices,
        'open': prices * 0.999,
        'high': prices * 1.01,
        'low': prices * 0.99,
        'volume': np.random.lognormal(8, 0.5, n)
    }, index=dates)
else:
    print(f"✅ 成功獲取 {len(btc_data)} 個數據點")

print(f"數據期間: {btc_data.index[0]} 到 {btc_data.index[-1]}")
print(f"價格範圍: ${btc_data['close'].min():,.0f} - ${btc_data['close'].max():,.0f}")

btc_data.head()

In [ ]:
# 繪製價格和收益率時間序列
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# 價格時間序列
axes[0].plot(btc_data.index, btc_data['close'], linewidth=1.5, color='blue')
axes[0].set_title('BTC 價格走勢 (4小時)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('價格 (USD)')
axes[0].grid(True, alpha=0.3)

# 計算並繪製收益率
returns = btc_data['close'].pct_change().dropna() * 100
axes[1].plot(returns.index, returns, linewidth=0.8, alpha=0.8, color='red')
axes[1].set_title('BTC 收益率 (4小時)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('收益率 (%)')
axes[1].set_xlabel('時間')
axes[1].grid(True, alpha=0.3)

# 添加零線
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# 收益率基本統計
print("=== BTC 收益率統計 ===")
print(f"平均收益率: {returns.mean():.4f}%")
print(f"波動率: {returns.std():.4f}%")
print(f"最大收益率: {returns.max():.4f}%")
print(f"最小收益率: {returns.min():.4f}%")

## 2. GARCH 模型分析

現在我們使用 GARCHModel 類來進行完整的波動率分析。

In [ ]:
# 初始化 GARCH 模型
garch_model = GARCHModel(btc_data, price_column='close')

print("GARCH 模型初始化完成")

### 2.1 收益率序列特徵分析

首先分析收益率的統計特徵，包括正態性檢定、偏度、峰度等。

In [ ]:
# 分析收益率特徵
returns_stats = garch_model.analyze_returns()

# 創建收益率分佈圖
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. 收益率直方圖與正態分佈比較
returns = garch_model.returns
axes[0].hist(returns, bins=50, density=True, alpha=0.7, color='skyblue', label='實際分佈')

# 疊加正態分佈
x = np.linspace(returns.min(), returns.max(), 100)
normal_curve = stats.norm.pdf(x, returns.mean(), returns.std())
axes[0].plot(x, normal_curve, 'r-', linewidth=2, label='正態分佈')
axes[0].set_title('收益率分佈 vs 正態分佈')
axes[0].set_xlabel('收益率 (%)')
axes[0].set_ylabel('密度')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Q-Q 圖
from scipy import stats
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title('收益率 Q-Q 圖')
axes[1].grid(True, alpha=0.3)

# 3. 收益率絕對值 (波動率代理)
abs_returns = np.abs(returns)
axes[2].plot(abs_returns.index, abs_returns, linewidth=0.8, alpha=0.7, color='green')
axes[2].set_title('絕對收益率 (波動率代理)')
axes[2].set_xlabel('時間')
axes[2].set_ylabel('絕對收益率 (%)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 ARCH 效應檢測

檢測收益率序列中是否存在 ARCH 效應（異方差性），這是使用 GARCH 模型的前提。

In [ ]:
# 檢測 ARCH 效應
arch_results = garch_model.test_arch_effects()

# 視覺化 ARCH 效應
returns = garch_model.returns
squared_returns = returns ** 2

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 收益率時間序列
axes[0, 0].plot(returns.index, returns, linewidth=0.8, alpha=0.7)
axes[0, 0].set_title('收益率時間序列')
axes[0, 0].set_ylabel('收益率 (%)')
axes[0, 0].grid(True, alpha=0.3)

# 2. 收益率平方（波動率代理）
axes[0, 1].plot(squared_returns.index, squared_returns, linewidth=0.8, alpha=0.7, color='red')
axes[0, 1].set_title('收益率平方 (波動率代理)')
axes[0, 1].set_ylabel('收益率平方 (%²)')
axes[0, 1].grid(True, alpha=0.3)

# 3. 滾動標準差
rolling_vol = returns.rolling(window=20).std()
axes[1, 0].plot(rolling_vol.index, rolling_vol, linewidth=1, color='purple')
axes[1, 0].set_title('20期滾動波動率')
axes[1, 0].set_ylabel('滾動波動率 (%)')
axes[1, 0].grid(True, alpha=0.3)

# 4. 收益率平方的自相關函數
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(squared_returns.dropna(), lags=20, ax=axes[1, 1], alpha=0.05)
axes[1, 1].set_title('收益率平方的自相關函數')

plt.tight_layout()
plt.show()

# 解釋 ARCH 測試結果
if arch_results['has_arch_effects']:
    print("\n🎯 ARCH 效應檢測結果: 檢測到顯著的 ARCH 效應")
    print("   ✅ 適合使用 GARCH 模型進行波動率建模")
    print("   📊 波動率呈現聚集性特徵")
else:
    print("\n⚠️  ARCH 效應檢測結果: 未檢測到顯著的 ARCH 效應")
    print("   📝 GARCH 模型效果可能有限，但仍可嘗試")

### 2.3 GARCH 模型擬合

擬合 GARCH(1,1) 模型並分析參數估計結果。

In [ ]:
# 擬合 GARCH(1,1) 模型
fitted_model = garch_model.fit_garch(p=1, q=1, mean_model='constant', vol_model='GARCH')

# 分析模型參數
params = fitted_model.params
print("\n=== GARCH(1,1) 模型參數分析 ===")
print(f"ω (omega): {params['omega']:.6f} - 長期波動率水準")
if 'alpha[1]' in params.index:
    print(f"α (alpha): {params['alpha[1]']:.6f} - 短期衝擊影響")
if 'beta[1]' in params.index:
    print(f"β (beta):  {params['beta[1]']:.6f} - 波動率持續性")

# 計算持續性指標
if 'alpha[1]' in params.index and 'beta[1]' in params.index:
    persistence = params['alpha[1]'] + params['beta[1]']
    print(f"\n持續性指標 (α + β): {persistence:.6f}")
    if persistence < 1:
        print("✅ 模型滿足平穩性條件")
        print(f"長期無條件波動率: {np.sqrt(params['omega'] / (1 - persistence)):.4f}%")
    else:
        print("⚠️  模型接近單位根，波動率衝擊影響很持久")

print(f"\n模型擬合品質:")
print(f"對數似然值: {fitted_model.loglikelihood:.2f}")
print(f"AIC: {fitted_model.aic:.2f}")
print(f"BIC: {fitted_model.bic:.2f}")

### 2.4 模型診斷

進行模型診斷，檢查模型適合度和殘差性質。

In [ ]:
# 模型診斷
garch_model.model_diagnostics()

# 創建診斷圖表
garch_model.plot_volatility_analysis()

print("\n📋 模型診斷要點:")
print("1. 參數顯著性: 所有參數應該在統計上顯著 (p值 < 0.05)")
print("2. 殘差性質: 標準化殘差應該接近白噪音")
print("3. ARCH 效應: 殘差中不應該存在剩餘的 ARCH 效應")
print("4. 正態性: 標準化殘差應該近似正態分佈")

### 2.5 波動率預測

使用擬合好的 GARCH 模型進行波動率預測。

In [ ]:
# 進行波動率預測
forecast_result = garch_model.forecast_volatility(horizon=24, method='simulation')

# 創建詳細的預測結果表格
vol_forecast = forecast_result['volatility_forecast']
forecast_df = pd.DataFrame({
    '預測波動率(%)': vol_forecast,
    '預測方差': forecast_result['variance_forecast']
})

print("\n=== 未來24期波動率預測 ===")
print(forecast_df.head(10).round(4))

# 波動率預測統計
current_vol = np.sqrt(fitted_model.conditional_volatility.iloc[-1])
avg_forecast_vol = vol_forecast.mean()
max_forecast_vol = vol_forecast.max()
min_forecast_vol = vol_forecast.min()

print(f"\n📊 波動率預測統計:")
print(f"當前波動率: {current_vol:.4f}%")
print(f"平均預測波動率: {avg_forecast_vol:.4f}%")
print(f"預測波動率範圍: {min_forecast_vol:.4f}% - {max_forecast_vol:.4f}%")

# 波動率趨勢分析
vol_change = ((avg_forecast_vol - current_vol) / current_vol) * 100
print(f"預測波動率變化: {vol_change:+.2f}%")

if vol_change > 5:
    print("📈 預測波動率將顯著上升")
elif vol_change < -5:
    print("📉 預測波動率將顯著下降")
else:
    print("➡️  預測波動率將保持相對穩定")

In [ ]:
# 創建波動率預測視覺化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 歷史條件波動率 + 預測
hist_vol = fitted_model.conditional_volatility
forecast_vol = forecast_result['volatility_forecast']

# 顯示最近50期歷史 + 預測
axes[0, 0].plot(hist_vol.index[-50:], hist_vol.iloc[-50:], 'b-', linewidth=2, label='歷史波動率')
axes[0, 0].plot(forecast_vol.index, forecast_vol.values, 'r--', linewidth=2, label='預測波動率')
axes[0, 0].set_title('波動率預測')
axes[0, 0].set_ylabel('波動率 (%)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 預測波動率分佈
axes[0, 1].hist(forecast_vol.values, bins=20, alpha=0.7, color='orange')
axes[0, 1].axvline(forecast_vol.mean(), color='red', linestyle='--', linewidth=2, label=f'平均值: {forecast_vol.mean():.4f}%')
axes[0, 1].set_title('預測波動率分佈')
axes[0, 1].set_xlabel('波動率 (%)')
axes[0, 1].set_ylabel('頻率')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 波動率預測趨勢
forecast_trend = (forecast_vol.values / forecast_vol.iloc[0] - 1) * 100
axes[1, 0].plot(range(len(forecast_trend)), forecast_trend, 'g-o', linewidth=2, markersize=4)
axes[1, 0].set_title('波動率預測變化趨勢')
axes[1, 0].set_xlabel('預測期數')
axes[1, 0].set_ylabel('變化率 (%)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)

# 4. 歷史波動率統計
vol_stats = [
    hist_vol.mean(),
    hist_vol.std(), 
    hist_vol.min(),
    hist_vol.max(),
    forecast_vol.mean()
]
vol_labels = ['歷史平均', '歷史標準差', '歷史最小', '歷史最大', '預測平均']
colors = ['blue', 'lightblue', 'green', 'red', 'orange']

bars = axes[1, 1].bar(vol_labels, vol_stats, color=colors, alpha=0.7)
axes[1, 1].set_title('波動率統計比較')
axes[1, 1].set_ylabel('波動率 (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

# 添加數值標籤
for bar, stat in zip(bars, vol_stats):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{stat:.3f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### 2.6 風險指標計算

使用 GARCH 模型預測的波動率來計算 VaR (風險價值) 和 CVaR (條件風險價值)。

In [ ]:
# 計算不同信心水準的風險指標
risk_metrics_1d = garch_model.calculate_var_cvar(confidence_levels=[0.90, 0.95, 0.99], horizon=1)
risk_metrics_7d = garch_model.calculate_var_cvar(confidence_levels=[0.90, 0.95, 0.99], horizon=7)

print("\n=== 7日風險指標 ===")
for key, value in risk_metrics_7d.items():
    if key.startswith('VaR_') or key.startswith('CVaR_'):
        print(f"{key}: {value:.4f}%")

# 創建風險指標視覺化
confidence_levels = [90, 95, 99]
var_1d = [risk_metrics_1d[f'VaR_{level}'] for level in confidence_levels]
cvar_1d = [risk_metrics_1d[f'CVaR_{level}'] for level in confidence_levels]
var_7d = [risk_metrics_7d[f'VaR_{level}'] for level in confidence_levels]
cvar_7d = [risk_metrics_7d[f'CVaR_{level}'] for level in confidence_levels]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1日風險指標
x = np.arange(len(confidence_levels))
width = 0.35

axes[0].bar(x - width/2, np.abs(var_1d), width, label='VaR', alpha=0.8, color='lightcoral')
axes[0].bar(x + width/2, np.abs(cvar_1d), width, label='CVaR', alpha=0.8, color='lightblue')
axes[0].set_title('1日風險指標')
axes[0].set_ylabel('風險值 (%)')
axes[0].set_xlabel('信心水準')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{level}%' for level in confidence_levels])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 7日風險指標
axes[1].bar(x - width/2, np.abs(var_7d), width, label='VaR', alpha=0.8, color='lightcoral')
axes[1].bar(x + width/2, np.abs(cvar_7d), width, label='CVaR', alpha=0.8, color='lightblue')
axes[1].set_title('7日風險指標')
axes[1].set_ylabel('風險值 (%)')
axes[1].set_xlabel('信心水準')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{level}%' for level in confidence_levels])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 風險解釋
var_95_7d = abs(risk_metrics_7d['VaR_95'])
print(f"\n🎯 風險指標解讀:")
print(f"95% 信心水準下的 7日 VaR: {var_95_7d:.2f}%")
print(f"   ➡️  在未來7天中，有95%的概率損失不會超過 {var_95_7d:.2f}%")
print(f"   ➡️  換句話說，只有5%的概率損失會超過這個數值")

current_price = btc_data['close'].iloc[-1]
dollar_risk = current_price * var_95_7d / 100
print(f"   💰 以當前價格 ${current_price:,.0f} 計算，風險金額約為 ${dollar_risk:,.0f}")

### 2.7 實際應用案例

展示 GARCH 模型在實際交易和風險管理中的應用。

In [ ]:
# 實際應用案例：動態倉位管理
print("=== GARCH 模型實際應用案例 ===")

# 案例1：基於波動率的倉位調整
current_vol = np.sqrt(fitted_model.conditional_volatility.iloc[-1])
avg_vol = fitted_model.conditional_volatility.mean()
vol_ratio = current_vol / avg_vol

print(f"\n📈 案例1：動態倉位管理")
print(f"當前波動率: {current_vol:.4f}%")
print(f"歷史平均波動率: {avg_vol:.4f}%")
print(f"波動率比率: {vol_ratio:.2f}")

base_position = 1.0  # 基準倉位
adjusted_position = base_position / vol_ratio  # 反向調整

print(f"\n建議倉位調整:")
if vol_ratio > 1.2:
    print(f"🔻 當前波動率較高，建議降低倉位至 {adjusted_position:.2f}")
elif vol_ratio < 0.8:
    print(f"🔺 當前波動率較低，可考慮增加倉位至 {adjusted_position:.2f}")
else:
    print(f"➡️  當前波動率正常，維持標準倉位 {base_position:.2f}")

# 案例2：期權定價輔助
print(f"\n📊 案例2：期權定價輔助")
forecast_vol = forecast_result['volatility_forecast'].mean()
historical_vol = fitted_model.conditional_volatility.rolling(30).mean().iloc[-1]

print(f"30日歷史平均波動率: {historical_vol:.4f}%")
print(f"GARCH 預測波動率: {forecast_vol:.4f}%")
print(f"波動率差異: {((forecast_vol/historical_vol)-1)*100:+.2f}%")

if forecast_vol > historical_vol * 1.1:
    print("💡 GARCH 預測波動率上升，期權可能被低估")
elif forecast_vol < historical_vol * 0.9:
    print("💡 GARCH 預測波動率下降，期權可能被高估")
else:
    print("💡 預測波動率與歷史水準接近")

# 案例3：風險預警系統
print(f"\n⚠️  案例3：風險預警系統")
vol_percentile = (current_vol <= fitted_model.conditional_volatility).mean() * 100
print(f"當前波動率在歷史分佈中的百分位: {vol_percentile:.1f}%")

if vol_percentile >= 90:
    risk_level = "🔴 高風險"
    recommendation = "建議降低槓桿、增加對沖"
elif vol_percentile >= 70:
    risk_level = "🟡 中等風險"
    recommendation = "保持謹慎、密切監控"
else:
    risk_level = "🟢 低風險"
    recommendation = "可考慮適度增加曝險"

print(f"風險等級: {risk_level}")
print(f"操作建議: {recommendation}")

## 3. 總結與下一步

### 3.1 GARCH 模型總結

通過本次分析，我們學會了：
1. **GARCH 模型原理**：理解波動率聚集和異方差性
2. **模型實作流程**：從數據準備到模型診斷
3. **波動率預測**：預測未來波動率變化
4. **風險管理**：計算 VaR 和 CVaR 風險指標
5. **實際應用**：動態倉位管理和風險預警

### 3.2 模型限制
- **參數穩定性**：模型參數可能隨時間變化
- **分佈假設**：假設誤差項為正態分佈
- **線性限制**：無法捕捉非線性波動率模式
- **外部因素**：無法直接納入市場事件影響

### 3.3 改進方向
- **EGARCH 模型**：處理波動率不對稱性
- **多變量 GARCH**：分析多資產波動率關聯
- **機器學習**：使用 LSTM 等方法建模波動率
- **實時更新**：建立動態模型更新機制

In [ ]:
# 保存模型結果
print("=== 模型結果保存 ===")

# 保存預測結果
forecast_df = pd.DataFrame({
    '預測波動率': forecast_result['volatility_forecast'],
    '預測方差': forecast_result['variance_forecast']
})

forecast_df.to_csv('../../results/model_outputs/garch_volatility_forecast.csv')
print("📊 波動率預測結果已保存")

# 保存風險指標
import json
risk_summary = {
    'model_type': 'GARCH(1,1)',
    'data_period': f"{btc_data.index[0]} to {btc_data.index[-1]}",
    'current_volatility': float(current_vol),
    'forecast_horizon': 24,
    'avg_forecast_volatility': float(forecast_vol.mean()),
    'var_95_1d': float(risk_metrics_1d['VaR_95']),
    'cvar_95_1d': float(risk_metrics_1d['CVaR_95']),
    'var_95_7d': float(risk_metrics_7d['VaR_95']),
    'cvar_95_7d': float(risk_metrics_7d['CVaR_95']),
    'model_aic': float(fitted_model.aic),
    'model_bic': float(fitted_model.bic)
}

with open('../../results/model_outputs/garch_risk_summary.json', 'w', encoding='utf-8') as f:
    json.dump(risk_summary, f, indent=2, ensure_ascii=False)

print("📋 風險指標摘要已保存")
print("\n✅ GARCH 模型分析完成！")
print("\n🎯 下一步學習計劃:")
print("1. 學習 GBM 模型進行價格路徑模擬")
print("2. 結合 ARIMA + GARCH 建立完整預測系統")
print("3. 開發基於波動率的交易策略")
print("4. 探索機器學習波動率預測方法")